# Individual method runner

Run ONE TALENT method on ONE credit dataset through the full wrapper pipeline (folds, caps, metric enrichment) — the quickest way to sanity-check a method, a dataset, or a fresh TALENT install without touching SLURM or the saved results.

In [ ]:
import sys, time
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.runtime_quiet import configure_quiet_runtime
configure_quiet_runtime()
from src.methods.method_runner import run_talent_method
from src.methods.method_config import HPO_METHODS, METHOD_ROW_LIMITS

In [ ]:
DATASET = '0001.gmsc'   # any stem under data/raw/<task>/
METHOD  = 'tabpfn_v3'   # any TALENT registry name (tabpfncredit list)
TASK    = 'pd'          # 'pd' (classification) or 'lgd' (regression)
TUNE    = False         # True only for methods in HPO_METHODS

print('tunable:', METHOD in HPO_METHODS, '| train cap:', METHOD_ROW_LIMITS.get(METHOD, 'none'))
t0 = time.time()
fold_results = run_talent_method(task=TASK, dataset=DATASET, method=METHOD,
                                 cv_splits=2, tune=TUNE, verbose=True)
print(f'{time.time()-t0:.1f}s total')

In [ ]:
import pandas as pd
metrics = pd.DataFrame({f'fold {k}': v['metrics'] for k, v in fold_results.items()})
metrics['mean'] = metrics.mean(axis=1)
display(metrics.round(4))

Nothing is written to `results/` — this runner is for inspection only. To produce a real result file, use `tabpfncredit experiment <name> --dataset ... --method ...`.